In [3]:
# 1. Install missing Colab libraries
!pip install -q transformers pandas scikit-learn

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix
import os

# 2. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# --- CONFIGURATION (Change these paths if needed) ---
# Assuming your dataset is in a folder called "CS4248_PROJ" in your main Drive
DATA_PATH = "/content/drive/MyDrive/CS4248/test_hf_model/annotated_elon_tweets.csv"
OUTPUT_PATH = "/content/drive/MyDrive/CS4248/test_hf_model/cardiff_roberta.csv"
TEXT_COLUMN = "clean_text"
LABEL_COLUMN = "label"
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [7]:
class SentimentDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }

print("Checking for dataset...")
if not os.path.exists(DATA_PATH):
    print(f"ERROR: Could not find dataset at {DATA_PATH}")
    print("Please check your Drive path!")
else:
    print("Found dataset! Loading model and forcing Colab GPU...")
    # Force Google Colab's Free CUDA GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Colab Hardware Backend: {device}")

    # Fetch Cardiff model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model.to(device)
    model.eval()

    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COLUMN].values

    # Dataloader for fast batching
    dataset = SentimentDataset(texts, tokenizer)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

    predictions = []
    probabilities = []

    print("Running Hardware-Accelerated Inference...")
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)

            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(outputs.logits, dim=-1)

            predictions.extend(preds.cpu().numpy())
            probabilities.extend(probs.cpu().numpy())

    print(f"\nProcessing Complete. Saving to {OUTPUT_PATH}")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    out_df = pd.DataFrame({"pred": predictions})

    probs_array = np.array(probabilities)
    out_df["prob_negative"] = probs_array[:, 0]
    out_df["prob_neutral"] = probs_array[:, 1]
    out_df["prob_positive"] = probs_array[:, 2]
    out_df["confidence"] = probs_array.max(axis=1)

    out_df.to_csv(OUTPUT_PATH, index=False)
    print("Saved successfully!")

    if LABEL_COLUMN in df.columns:
        print("\n" + "="*50)
        print("  CardiffNLP Performance Metrics")
        print("="*50)
        gold = df[LABEL_COLUMN].values
        valid = ~pd.isna(gold)
        y_true = gold[valid].astype(int)
        y_pred = np.array(predictions)[valid]
        print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"], zero_division=0))
        print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


Checking for dataset...
Found dataset! Loading model and forcing Colab GPU...
Colab Hardware Backend: cpu


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running Hardware-Accelerated Inference...

Processing Complete. Saving to /content/drive/MyDrive/CS4248/test_hf_model/cardiff_roberta.csv
Saved successfully!

  CardiffNLP Performance Metrics
              precision    recall  f1-score   support

    negative       0.66      0.74      0.70        61
     neutral       0.78      0.81      0.80       217
    positive       0.78      0.70      0.74       122

    accuracy                           0.76       400
   macro avg       0.74      0.75      0.74       400
weighted avg       0.76      0.76      0.76       400

Confusion matrix:
 [[ 45  14   2]
 [ 20 175  22]
 [  3  34  85]]
